# Phase 8 — Testing & Evaluation
**Goal:** Test on all datasets, compare models, show confusion matrices and full performance analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score, roc_curve)

xgb_ulb  = joblib.load('../models/xgboost_model.pkl')
xgb_ieee = joblib.load('../models/xgboost_ieee_model.pkl')
lr_model = joblib.load('../models/logistic_model.pkl')
rf_model = joblib.load('../models/random_forest_model.pkl')
X_test   = joblib.load('../data/X_test.pkl')
y_test   = joblib.load('../data/y_test.pkl')
print("All models and data loaded!")

In [ ]:
# Full comparison on ULB test set
models = [
    ('Logistic Regression', lr_model),
    ('Random Forest',       rf_model),
    ('XGBoost',             xgb_ulb),
]

rows = []
for name, model in models:
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:,1]
    rows.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred),4),
        'Precision': round(precision_score(y_test, y_pred),4),
        'Recall'   : round(recall_score(y_test, y_pred),4),
        'F1'       : round(f1_score(y_test, y_pred),4),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_pred_prob),4),
    })

results_df = pd.DataFrame(rows).set_index('Model')
print("=== FULL MODEL COMPARISON (ULB Dataset) ===")
print(results_df.to_string())
print("\nBest Recall :", results_df['Recall'].idxmax())
print("Best F1     :", results_df['F1'].idxmax())
print("Best ROC-AUC:", results_df['ROC-AUC'].idxmax())

In [ ]:
# Confusion matrices — all 3 models side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Fraud'], yticklabels=['Normal','Fraud'])
    ax.set_title(f'{name}')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices — All Models (ULB Dataset)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/phase8_all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves — all models
plt.figure(figsize=(8,6))
for name, model in models:
    prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', lw=2)
plt.plot([0,1],[0,1],'k--',lw=1,label='Random baseline')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models'); plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/phase8_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Multi-dataset summary
ledger = pd.read_csv('../data/fraud_ledger.csv')
print("=== FRAUD LEDGER SUMMARY ===")
print(f"Total decisions logged : {len(ledger)}")
print("\nBy dataset source:")
print(ledger.groupby('dataset_source')['action'].value_counts().to_string())
print("\nAction distribution:")
print(ledger['action'].value_counts())

In [ ]:
# Final summary chart
metrics = ['Accuracy','Precision','Recall','F1','ROC-AUC']
x = np.arange(len(metrics)); width = 0.25
fig, ax = plt.subplots(figsize=(13,6))
colors = ['steelblue','seagreen','crimson']
for i,(name,_) in enumerate(models):
    vals = results_df.loc[name, metrics].values
    bars = ax.bar(x + (i-1)*width, vals, width, label=name, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0,1.12); ax.set_ylabel('Score')
ax.set_title('Final Model Comparison — All Metrics', fontsize=13)
ax.legend(); ax.axhline(y=0.9,color='gray',linestyle='--',alpha=0.4)
plt.tight_layout()
plt.savefig('../reports/phase8_final_comparison.png', dpi=150)
plt.show()
print("Phase 8 complete! All reports saved.")